# Phase 6: Backtest, Dual Bound & P&L Attribution

Run asset/degradation checks, dual-bound verification, synthetic backtest, and P&L attribution. This notebook is generated from `bess_valuation_full.ipynb` Phase 6 and can run independently from saved processed artefacts, with mini fallbacks for development.


In [ ]:
import sys, json, warnings
from pathlib import Path

# Bootstrap imports from the project root, then use the shared helper.
_ROOT_CANDIDATE = Path.cwd()
for candidate in [_ROOT_CANDIDATE, *_ROOT_CANDIDATE.parents]:
    if (candidate / "src").is_dir() and (candidate / "data").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not find project root containing src/ and data/.")

from src.utils import find_project_root

PROJECT_ROOT = find_project_root(_ROOT_CANDIDATE)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from src.config import ASSET, FINANCE, DEGRADATION, LSMC, VALIDATION, configure_asset_duration
from src.asset.battery import BatteryAsset, initial_state
from src.asset.degradation import DegradationModel
from src.attribution.pnl_explain import (
    PnlExplainer, FactorChanges, backtest_summary, print_backtest_summary,
)

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

# Valuation duration selector: choose 1, 2, or 4 hours.
# Phase 6 reads Phase 4/5 artefacts generated with the same duration.
VALUATION_DURATION_H = float(globals().get("VALUATION_DURATION_H", 2.0))
configure_asset_duration(ASSET, VALUATION_DURATION_H)
DURATION_LABEL = f"{ASSET['duration_h']:g}h"
print(f"Asset under valuation: {ASSET['power_mw']:.0f} MW / {ASSET['energy_mwh']:.0f} MWh ({ASSET['duration_h']:.0f}h)")
print(f"Output label: {DURATION_LABEL}")

print(f'Project root: {PROJECT_ROOT}')
print(f'Processed data dir: {PROCESSED}')


## 1. Battery Asset & Degradation

In [ ]:
battery   = BatteryAsset(ASSET)
deg_model = DegradationModel(DEGRADATION, ASSET)
print(battery.summary())

# Shadow price at SoH levels
for soh in [1.0, 0.95, 0.90, 0.85, 0.82]:
    lam = deg_model.shadow_price(soh)
    print(f"  SoH={soh:.2f} -> lambda_deg = GBP {lam:.2f}/MWh")


In [ ]:
# 15-year SoH trajectory
years_base, soh_base = deg_model.simulate_soh_trajectory(
    efc_per_year=520, life_years=15, avg_soc_frac=0.50,
    augment_years=ASSET["augment_years"])
years_hi,   soh_hi   = deg_model.simulate_soh_trajectory(
    efc_per_year=730, life_years=15, avg_soc_frac=0.55,
    augment_years=ASSET["augment_years"])
years_lo,   soh_lo   = deg_model.simulate_soh_trajectory(
    efc_per_year=400, life_years=15, avg_soc_frac=0.45,
    augment_years=ASSET["augment_years"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years_base, soh_base*100, "b-",  lw=2,   label="Base (520 EFC/yr)")
ax.plot(years_hi,   soh_hi*100,   "r--", lw=1.5, label="High (730 EFC/yr, KYOS)")
ax.plot(years_lo,   soh_lo*100,   "g--", lw=1.5, label="Low  (400 EFC/yr)")
trig = ASSET["soh_augment_trigger"]
ax.axhline(trig*100, color="orange", lw=1.5, ls=":", label=f"Augment trigger ({trig:.0%})")
for yr in ASSET["augment_years"]: ax.axvline(yr, color="gray", lw=0.8, ls="--", alpha=0.6)
ax.set_xlabel("Year"); ax.set_ylabel("SoH (%)")
ax.set_title(f"SoH Trajectory ? {ASSET['energy_mwh']:.0f} MWh LFP BESS | Augmentations yr 4/8/12", fontweight="bold")
ax.legend(); ax.set_ylim(50, 105); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PROCESSED / f"soh_trajectory_{DURATION_LABEL}.png", dpi=150, bbox_inches="tight")
plt.show()


## 2. Dual Bound Verification

In [ ]:
POLICY_PATH = PROCESSED / f"lsmc_policy_{DURATION_LABEL}.pkl"
RESULT_PATH = PROCESSED / f"lsmc_valuation_result_{DURATION_LABEL}.pkl"
BUNDLE_PATH = PROCESSED / "sim_bundle.pkl"

if all(p.exists() for p in [POLICY_PATH, RESULT_PATH, BUNDLE_PATH]):
    import pickle
    with open(POLICY_PATH, "rb") as f: policy = pickle.load(f)
    with open(RESULT_PATH, "rb") as f: val_result = pickle.load(f)
    with open(BUNDLE_PATH, "rb") as f: bundle = pickle.load(f)
    print(f"Loaded: {len(val_result.pv_paths):,} paths")
else:
    print("Phase-4 artefacts not found -- running mini LSMC")
    from src.processes.simulate import simulate, default_params_from_config
    from src.optimisation.lsmc import run_lsmc
    from src.config import SCHWARTZ_SMITH

    ss_p, hpfc_p, imb_p, anc_p = default_params_from_config()
    lsmc_dev = dict(LSMC)
    lsmc_dev["n_paths"] = 100
    lsmc_dev["n_steps"] = 96
    xi_init = np.full(100, np.log(SCHWARTZ_SMITH["forward_anchor_gbp_mwh"]))
    bundle = simulate(
        ss_p, hpfc_p, imb_p, anc_p,
        n_paths=100, n_steps=96, dt=0.5/8760,
        xi_0=xi_init,
    )
    policy, val_result = run_lsmc(bundle, ASSET, lsmc_dev, DEGRADATION, FINANCE, hpfc_params=hpfc_p)


In [ ]:
from src.optimisation.dual_bound import compute_dual_bound

dual_result = compute_dual_bound(
    bundle=bundle, policy=policy, val_result=val_result,
    asset_cfg=ASSET, lsmc_cfg=LSMC, deg_cfg=DEGRADATION, fin_cfg=FINANCE,
    n_dual_paths=100, threshold=LSMC["dual_gap_acceptable"], verbose=True,
)

DUAL_HORIZON_STEPS = min(int(policy.n_steps), int(bundle.n_steps), int(val_result.cashflow_paths.shape[1]))
DUAL_HORIZON_YEARS = DUAL_HORIZON_STEPS * float(policy.dt_h) / 8760.0
DUAL_ANNUALIZATION_FACTOR = 1.0 / DUAL_HORIZON_YEARS if DUAL_HORIZON_YEARS > 0 else float("nan")
DUAL_POWER_MW = float(ASSET["power_mw"])
dual_v_lsmc_annual = dual_result.v_lsmc * DUAL_ANNUALIZATION_FACTOR
dual_v_dual_annual = dual_result.v_dual * DUAL_ANNUALIZATION_FACTOR
dual_v_dual_std_annual = dual_result.v_dual_std * DUAL_ANNUALIZATION_FACTOR
dual_gap_abs_annual = dual_result.gap_abs * DUAL_ANNUALIZATION_FACTOR


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
labels = ["V_LSMC annual (lower)", "V_dual annual (upper)"]
vals   = [dual_v_lsmc_annual, dual_v_dual_annual]
colors = ["#3498db", "#e74c3c"]
bars = ax.barh(labels, vals, color=colors, edgecolor="white", height=0.45)
for bar in bars:
    w = bar.get_width()
    ax.text(w*1.01, bar.get_y()+bar.get_height()/2, f"GBP {w:,.0f}/yr", va="center", fontsize=9)
gap_str = f"{dual_result.gap_pct:.2%}"
ok_str  = "PASS" if dual_result.dual_ok else "REFINE"
ax.set_xlabel("Annualized value (GBP/year)")
ax.set_title(f"LSMC Optimality Gap: {gap_str} ({ok_str}) | target < {dual_result.threshold:.0%}",
             fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(PROCESSED / f"dual_bound_{DURATION_LABEL}.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Synthetic 30-Day Backtest

In [ ]:
rng = np.random.default_rng(42)
N_DAYS = 30

mtm_base   = 50_000_000.0
daily_vol  = mtm_base * 0.008
mtm_series = np.zeros(N_DAYS + 1)
mtm_series[0] = mtm_base
for d in range(N_DAYS):
    mtm_series[d+1] = mtm_series[d] + rng.normal(0, daily_vol)

cf_expected = rng.normal(80_000, 15_000, N_DAYS)
cf_realised = cf_expected + rng.normal(0, 8_000, N_DAYS)

soh_model_daily  = np.linspace(1.000, 0.998, N_DAYS+1)
soh_actual_daily = np.clip(soh_model_daily + rng.normal(0, 0.0002, N_DAYS+1), 0.95, 1.0)

factor_changes = [
    FactorChanges(
        delta_baseload_gbp_mwh  = float(rng.normal(0, 1.5)),
        delta_vega_da_frac      = float(rng.normal(0, 0.02)),
        delta_dc_gbp_mwh        = float(rng.normal(0, 0.3)),
        delta_qr_gbp_mwh        = float(rng.normal(0, 0.2)),
        delta_imb_drift_gbp_mwh = float(rng.normal(0, 0.8)),
        delta_rho_bps           = float(rng.normal(0, 2.0)),
        delta_avail_pp          = float(rng.normal(0, 0.1)),
    ) for _ in range(N_DAYS)
]
print(f"Synthetic backtest: {N_DAYS} days ready")


In [ ]:
MTM_SUMMARY = PROCESSED / f"mtm_summary_{DURATION_LABEL}.json"
if MTM_SUMMARY.exists():
    with open(MTM_SUMMARY) as f:
        greeks_dict = json.load(f).get("greeks", {})
    print(f"Loaded {len(greeks_dict)} Greeks")
else:
    greeks_dict = {
        "delta_baseload": {"greek": 2_500_000.0, "bump_size": 1.0, "bump_unit": "GBP/MWh"},
        "delta_dc":       {"greek":   800_000.0, "bump_size": 1.0, "bump_unit": "GBP/MW/h"},
        "delta_qr":       {"greek":   400_000.0, "bump_size": 1.0, "bump_unit": "GBP/MW/h"},
        "vega_da":        {"greek": 3_000_000.0, "bump_size": 0.10,"bump_unit": "fraction"},
        "delta_imb_drift":{"greek":   600_000.0, "bump_size": 5.0, "bump_unit": "GBP/MWh"},
        "rho":            {"greek":-25_000_000.0,"bump_size":50.0, "bump_unit": "bps"},
        "delta_avail":    {"greek":-2_500_000.0, "bump_size":-2.0, "bump_unit": "pp"},
    }
    print("Using synthetic Greeks")

explainer = PnlExplainer(greeks_dict, ASSET, FINANCE)
results = explainer.explain_series(
    mtm_series=mtm_series, cf_realised=cf_realised, cf_expected=cf_expected,
    soh_actual=soh_actual_daily, soh_model=soh_model_daily,
    factor_changes=factor_changes,
)
results[0].print_waterfall()


In [ ]:
bk_summary = backtest_summary(results, VALIDATION)
print_backtest_summary(bk_summary)


## 4. Attribution Charts

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), height_ratios=[1.5, 1])

days   = np.arange(N_DAYS)
labels = ["Theta", "Delta-explain", "Exec surprise", "Deg surprise", "Residual"]
series = [
    np.array([r.theta for r in results]),
    np.array([r.total_delta_explain for r in results]),
    np.array([r.execution_surprise for r in results]),
    np.array([r.degradation_surprise for r in results]),
    np.array([r.residual for r in results]),
]
colors = ["#3498db", "#e74c3c", "#27ae60", "#f39c12", "#95a5a6"]

ax = axes[0]
bp = np.zeros(N_DAYS); bn = np.zeros(N_DAYS)
for s, lbl, col in zip(series, labels, colors):
    pos = np.where(s>0, s, 0); neg = np.where(s<0, s, 0)
    ax.bar(days, pos, bottom=bp, color=col, label=lbl, alpha=0.85)
    ax.bar(days, neg, bottom=bn, color=col, alpha=0.85)
    bp += pos; bn += neg
ax.axhline(0, color="#333", lw=0.8)
ax.set_ylabel("Daily ΔMTM (GBP)"); ax.set_title("Daily P&L Attribution", fontweight="bold")
ax.legend(loc="upper right", fontsize=8, ncol=3); ax.grid(axis="y", alpha=0.3)

ax2 = axes[1]
rpcts = [r.residual_pct*100 for r in results]
bcols = ["#e74c3c" if p>5 else "#27ae60" for p in rpcts]
ax2.bar(days, rpcts, color=bcols, edgecolor="white")
ax2.axhline(5, color="#e74c3c", lw=1.5, ls="--", label="5% target")
ax2.set_xlabel("Day"); ax2.set_ylabel("Residual (%)")
ax2.set_title("Daily Residual vs 5% Target", fontweight="bold")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PROCESSED / f"pnl_attribution_{DURATION_LABEL}.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Save Results

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)

phase6 = {
    "duration_h": ASSET["duration_h"],
    "duration_label": DURATION_LABEL,
    "asset_mw": ASSET["power_mw"],
    "asset_mwh": ASSET["energy_mwh"],
    "dual_bound": {
        "value_basis": "annualized_gbp_per_year",
        "annualization_factor": float(DUAL_ANNUALIZATION_FACTOR),
        "horizon_years": float(DUAL_HORIZON_YEARS),
        "asset_mw": float(DUAL_POWER_MW),
        "v_lsmc_gbp": float(dual_v_lsmc_annual),
        "v_dual_gbp": float(dual_v_dual_annual),
        "v_dual_std": float(dual_v_dual_std_annual),
        "v_ri_gbp": float(getattr(dual_result, "v_ri", 0.0) * DUAL_ANNUALIZATION_FACTOR),
        "v_lsmc_gbp_per_mw_year": float(dual_v_lsmc_annual / DUAL_POWER_MW),
        "v_dual_gbp_per_mw_year": float(dual_v_dual_annual / DUAL_POWER_MW),
        "v_lsmc_horizon_gbp": float(dual_result.v_lsmc),
        "v_dual_horizon_gbp": float(dual_result.v_dual),
        "gap_abs_gbp": float(dual_gap_abs_annual),
        "gap_pct":  dual_result.gap_pct,
        "n_paths": int(dual_result.n_paths),
        "dual_ok":  dual_result.dual_ok,
        "threshold": float(dual_result.threshold),
    },
    "backtest": bk_summary,
    "soh_base_yr15": float(soh_base[-1]),
}

out = PROCESSED / f"phase6_summary_{DURATION_LABEL}.json"
with open(out, "w") as f:
    json.dump(phase6, f, indent=2, default=str)

print(f"Saved: {out}")
print(f"  Dual gap: {dual_result.gap_pct:.2%}")
gap_ok = "PASS" if dual_result.dual_ok else "REFINE"
print(f"  Status:   {gap_ok}")
print(f"  Backtest residual: {bk_summary.get(chr(116)+chr(111)+chr(116)+chr(97)+chr(108)+chr(95)+chr(114)+chr(101)+chr(115)+chr(105)+chr(100)+chr(117)+chr(97)+chr(108)+chr(95)+chr(112)+chr(99)+chr(116), 0):.2%}")


---

## Model Complete ✓

All six phases have been executed end-to-end.  
Key artefacts written to `data/processed/`:

| File | Content |
|---|---|
| `ss_params.json` | Calibrated Schwartz-Smith parameters |
| `sim_bundle.pkl` | 5,000-path joint simulation bundle |
| `lsmc_policy.pkl` | LSMC regression coefficients β[t, j, k, 14] |
| `lsmc_valuation_result.pkl` | Forward-pass ValuationResult |
| `mtm_summary.json` | Full MTM decomposition + Greeks + VaR |
| `phase6_summary.json` | Dual bound gap + backtest attribution |
